# Phase 2: Data Cleaning and Preparation (Customer Churn)

**Goal of this notebook:** take the raw data from Phase 1 and produce two ready-to-use datasets, one readable for analysis and one encoded for modeling.

**Important point to say to the panel:** Phase 1 showed the data is already clean. So this phase is mostly *verifying* quality and *preparing* the data, not repairing it. We also write down what we would do if problems existed, because the panel may ask.

**Steps:**
1. Make a working copy of the raw data
2. Re-check missing values and duplicates
3. Check text columns for inconsistencies
4. Check outliers (IQR rule and boxplots)
5. Check logical consistency
6. Drop the ID column
7. Encode categorical columns
8. Final validation and save both datasets

## Step 1: Load the data and make a working copy
**What we do:** read the CSV and copy it into `df`.
**Why:** `raw` stays untouched, so if a cleaning step goes wrong we can start over. Never edit the original data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

raw = pd.read_csv("customer_churn_dataset-testing-master.csv")
df = raw.copy()
print("Shape:", df.shape)

## Step 2: Missing values and duplicates
**What we do:** count missing values per column and duplicate rows.
**Why:** most models cannot handle missing values, and duplicate rows give some customers extra weight.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerIDs:", df["CustomerID"].duplicated().sum())

**What we found:** no missing values and no duplicates, so nothing to remove.

**What we would do if we found problems (panel may ask):**

| Problem | Option | When to use it |
|---|---|---|
| Missing numeric value | Fill with the median | Median is not pulled by extreme values |
| Missing category | Fill with the most frequent value, or an "Unknown" label | Depends on how many are missing |
| Many missing values in a column (above about 40%) | Drop the column | The column carries too little information |
| Few rows with missing values | Drop those rows | Only if the loss is tiny |
| Duplicate rows | `drop_duplicates()` | Duplicates inflate some customers |

## Step 3: Check text columns
**What we do:** look at unique values and look for stray spaces or different spellings of the same thing (for example "male" and "Male").
**Why:** the model would treat "Male" and "male " as two different categories.

In [ ]:
categorical = ["Gender", "Subscription Type", "Contract Length"]

for c in categorical:
    spaces = (df[c] != df[c].str.strip()).sum()
    print(c, "| values:", df[c].unique().tolist(), "| rows with extra spaces:", spaces)

**What we found:** each categorical column has only its expected values and no spacing problems. No fix needed.

## Step 4: Outliers
**What we do:** apply the IQR rule. A value is an outlier if it is below Q1 - 1.5 x IQR or above Q3 + 1.5 x IQR. We also draw boxplots.
**Why:** extreme values can distort averages and some models. But an outlier is not automatically an error, so we check before removing anything.

In [ ]:
numeric = ["Age", "Tenure", "Usage Frequency", "Support Calls",
           "Payment Delay", "Total Spend", "Last Interaction"]

rows = []
for c in numeric:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[c] < low) | (df[c] > high)).sum()
    rows.append([c, q1, q3, low, high, n_out])

outlier_table = pd.DataFrame(rows, columns=["column", "Q1", "Q3", "lower_bound", "upper_bound", "outliers"])
outlier_table

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), numeric):
    ax.boxplot(df[c], vert=True)
    ax.set_title(c)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

**How to read a boxplot (say this to the panel):** the box covers the middle 50% of values (Q1 to Q3), the line inside is the median, and the whiskers extend to the IQR limits. Dots beyond the whiskers are outliers.

**What we found:** zero outliers in every numeric column. All values sit inside the IQR limits, and the skewness values are close to 0, so the distributions are roughly symmetric. We keep all rows.

**What we would do if outliers existed:**
- If it is a data error (age of 200), remove or correct it.
- If it is a real but extreme value (a very high spender), keep it, or cap it at the IQR limit (winsorizing). Removing real high-value customers would hide exactly the customers the business cares about.

## Step 5: Logical consistency checks
**What we do:** check that values make sense together, not just individually.
**Why:** a value can be in range but still impossible. For example, a customer cannot have been with the company longer than they have been alive.

In [ ]:
print("Tenure (months) longer than age in months:", (df["Tenure"] > df["Age"] * 12).sum())
print("Negative values in any numeric column:", (df[numeric] < 0).sum().sum())
print("Churn values:", sorted(df["Churn"].unique()))

**What we found:** no logical problems. Churn contains only 0 and 1.

## Step 6: Drop the ID column
**What we do:** remove `CustomerID` from the analysis data.
**Why:** it is unique for every row, so it says nothing about why customers leave. A model could memorise it and learn noise instead of patterns.

In [ ]:
df = df.drop(columns=["CustomerID"])
print("Shape after dropping CustomerID:", df.shape)
print(df.columns.tolist())

## Step 7: Encode categorical columns
**What we do:** convert the three text columns into numbers using one-hot encoding (`pd.get_dummies`).
**Why:** models work with numbers, not text. We do not use 0, 1, 2 for the categories because that would tell the model that Premium is "bigger" than Basic in a numeric sense. One-hot creates a 0/1 column per category instead.

**What `drop_first=True` does:** it removes one category per column. If Subscription Type has Basic, Standard and Premium, we keep only Standard and Premium. A customer with 0 in both is Basic, so no information is lost. This avoids a redundant column that would cause multicollinearity in models like logistic regression.

In [ ]:
df_encoded = pd.get_dummies(df, columns=categorical, drop_first=True, dtype=int)
print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

**What we found:** the 3 text columns became 5 numeric 0/1 columns (Gender_Male, Subscription Type_Premium, Subscription Type_Standard, Contract Length_Monthly, Contract Length_Quarterly). Baselines are Female, Basic and Annual.

**What we are NOT doing here, and why:** scaling (for example StandardScaler). Scaling must be fitted on the training data only, after the train/test split, otherwise information from the test set leaks into training. We do it inside the modeling phase (Phase 4).

## Step 8: Final validation and save
**What we do:** confirm the final datasets are clean, then save both.
**Why:** the next phases load these files directly, so we never repeat cleaning.

- `churn_clean.csv`: readable categories, no ID. Used for EDA (Phase 3) and the dashboard.
- `churn_model_ready.csv`: encoded numbers only. Used for modeling (Phase 4).

In [ ]:
assert df.isnull().sum().sum() == 0, "missing values found"
assert df.duplicated().sum() == 0, "duplicates found"
assert df_encoded.select_dtypes(exclude="number").shape[1] == 0, "non-numeric columns left"

df.to_csv("churn_clean.csv", index=False)
df_encoded.to_csv("churn_model_ready.csv", index=False)

print("churn_clean.csv       :", df.shape)
print("churn_model_ready.csv :", df_encoded.shape)
print("Churn rate preserved  :", round(df["Churn"].mean() * 100, 2), "%")

## Summary of Phase 2 (say this to the panel)
"The data was already clean: no missing values, no duplicates, no outliers by the IQR rule, and no logical inconsistencies. I verified each of these rather than assuming it. I dropped CustomerID because it is only an identifier, and one-hot encoded the three categorical columns so models can use them. I saved a readable version for analysis and an encoded version for modeling. I will do scaling in the modeling phase after the train/test split to avoid data leakage."

## Likely panel questions on this phase
- **Why did you not remove any outliers?** The IQR rule found none, and the skewness is near zero. If there were real extreme values, I would keep or cap them rather than delete, because they may be your most valuable customers.
- **How would you handle missing values?** Median for numeric columns (not affected by extremes), most frequent value for categories, and drop a column if most of it is missing.
- **Why one-hot encoding instead of 0, 1, 2?** Numeric labels imply an order that does not exist. One-hot treats each category as separate.
- **Why `drop_first=True`?** One column is redundant. Keeping it causes multicollinearity.
- **Why not scale now?** Scaling must be fitted on training data only. Doing it before the split leaks test information into training.
- **Why keep two datasets?** The readable one is for analysis and the dashboard. The encoded one is for models.

**Next:** Phase 3, EDA and statistical analysis in `data_eda.ipynb`.